# 3D Visualization of Trajectories

In this tutorial, we will use this notebook to visualize the 3D trajectories of two swimming adult zebrafish and will learn to interact with the trajectory data.

prepared for DNC2026 - Tatsuo Izawa

(Optional) To export the notebook to PDF from a terminal, run `jupyter nbconvert --to pdf 2fish_visualization.ipynb`.

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from IPython.display import Video, display
from matplotlib.animation import FFMpegWriter, FuncAnimation

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linewidth': 0.8,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.frameon': False,
})

PLOTLY_LAYOUT = dict(
    template='simple_white',
    font=dict(size=12),
    title_x=0.5,
    legend=dict(
        x=0.02,
        y=0.98,
        bgcolor='rgba(255,255,255,0.75)',
        bordercolor='rgba(0,0,0,0.1)',
        borderwidth=1,
    ),
    margin=dict(l=20, r=20, t=60, b=20),
)

COLOR_MAP = ['royalblue', 'crimson', 'seagreen', 'darkorange', 'mediumpurple']

file_path = Path('tracks_wt/Zebrafish20250204_0944-pc2.csv')
start_frame = 0
end_frame = 100
include_other_fish = True
body_part = 'pec'
fps = 140
SAVE_FIGURE = True  # Set to False to skip PNG export
GENERATE_MOVIE = True  # Set to False to skip MP4 export

print(f'Using file: {file_path}')
print(f'Frame range: {start_frame} to {end_frame}')
print(f'Body part: {body_part}')
print(f'Save figure: {SAVE_FIGURE}')

In [ ]:
def load_track_data(file_path, start_frame=0, end_frame=None, include_other_fish=False, body_part=body_part):
    track_data = {}
    missing_value_count = 0
    start_frame = max(start_frame, 0)
    end_frame = None if end_frame is None else max(end_frame, start_frame)

    with file_path.open(mode='r', newline='') as csvfile:
        reader = csv.reader(csvfile)
        header = next(reader)
        available_fish = sorted(
            {column.split('_')[0] for column in header[1:] if column.endswith(f'{body_part}_x')}
        )
        if not available_fish:
            raise ValueError(f'No fish columns found for body part: {body_part}')

        fish_to_plot = available_fish if include_other_fish else [available_fish[0]]
        column_indices = {
            fish_name: (
                header.index(f'{fish_name}_{body_part}_x'),
                header.index(f'{fish_name}_{body_part}_y'),
                header.index(f'{fish_name}_{body_part}_z'),
            )
            for fish_name in fish_to_plot
        }
        track_data = {fish_name: {'x': [], 'y': [], 'z': []} for fish_name in fish_to_plot}

        for frame_index, row in enumerate(reader):
            if frame_index < start_frame:
                continue
            if end_frame is not None and frame_index > end_frame:
                break
            for fish_name in fish_to_plot:
                x_idx, y_idx, z_idx = column_indices[fish_name]
                for axis_name, axis_idx in (('x', x_idx), ('y', y_idx), ('z', z_idx)):
                    raw_value = row[axis_idx].strip()
                    if raw_value == '':
                        track_data[fish_name][axis_name].append(np.nan)
                        missing_value_count += 1
                    else:
                        track_data[fish_name][axis_name].append(float(raw_value))

    for fish_name, coords in track_data.items():
        track_data[fish_name] = {
            axis: np.asarray(values, dtype=float)
            for axis, values in coords.items()
        }

    return track_data, missing_value_count


def axis_limits(values):
    values = np.asarray(values, dtype=float)
    finite_values = values[np.isfinite(values)]
    if finite_values.size == 0:
        return -0.5, 0.5
    value_min = finite_values.min()
    value_max = finite_values.max()
    if value_min == value_max:
        return value_min - 0.5, value_max + 0.5
    padding = (value_max - value_min) * 0.05
    return value_min - padding, value_max + padding


track_data, missing_value_count = load_track_data(
    file_path=file_path,
    start_frame=start_frame,
    end_frame=end_frame,
    include_other_fish=include_other_fish,
    body_part=body_part,)
fish_names = list(track_data.keys())
frame_count = len(next(iter(track_data.values()))['x'])
print(f'Loaded {frame_count} frames for {len(fish_names)} fish: {fish_names}')
print(f'Converted {missing_value_count} blank coordinate values to NaN.')

In [ ]:
def plot_fish_tracks(track_data, start_frame, end_frame, body_part='pec'):
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=coords['x'],
                y=coords['y'],
                z=coords['z'],
                mode='lines',
                line=dict(width=5, color=COLOR_MAP[index % len(COLOR_MAP)]),
                name=fish_name,
            )
            for index, (fish_name, coords) in enumerate(track_data.items())
        ]
    )

    fig.update_layout(
        **PLOTLY_LAYOUT,
        title=f'{body_part.title()} Tracks from Frame {start_frame} to {end_frame}',
        width=1000,
        height=800,
        scene=dict(
            xaxis_title='X [cm]',
            yaxis_title='Y [cm]',
            zaxis_title='Z [cm]',
            xaxis=dict(backgroundcolor='white', gridcolor='lightgray', zerolinecolor='lightgray'),
            yaxis=dict(backgroundcolor='white', gridcolor='lightgray', zerolinecolor='lightgray'),
            zaxis=dict(backgroundcolor='white', gridcolor='lightgray', zerolinecolor='lightgray'),
            domain=dict(x=[0.1, 0.9], y=[0.1, 0.9]),
            aspectmode='data',
        ),
    )
    fig.show()

def create_track_exports(track_data, source_file_path, start_frame, end_frame, body_part='pec', fps=140, output_path=None, save_figure=False, figure_output_path=None, export_movie=True):
    source_path = Path(source_file_path)
    if output_path is None:
        output_path = source_path.with_name(f'{source_path.stem}_{body_part}_tracks_{fps}fps.mp4')
    else:
        output_path = Path(output_path)

    if figure_output_path is None:
        figure_output_path = source_path.with_name(f'{source_path.stem}_{body_part}_tracks_snapshot.png')
    else:
        figure_output_path = Path(figure_output_path)

    all_x = np.concatenate([coords['x'] for coords in track_data.values()])
    all_y = np.concatenate([coords['y'] for coords in track_data.values()])
    all_z = np.concatenate([coords['z'] for coords in track_data.values()])

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(*axis_limits(all_x))
    ax.set_ylim(*axis_limits(all_y))
    ax.set_zlim(*axis_limits(all_z))
    ax.set_xlabel('X [cm]')
    ax.set_ylabel('Y [cm]')
    ax.set_zlabel('Z [cm]')
    ax.set_title(f'{body_part.title()} Tracks from Frame {start_frame} to {end_frame}')

    line_artists = []
    point_artists = []
    for index, (fish_name, coords) in enumerate(track_data.items()):
        color = COLOR_MAP[index % len(COLOR_MAP)]
        (line,) = ax.plot([], [], [], color=color, linewidth=2.5, label=fish_name)
        (point,) = ax.plot([], [], [], marker='o', color=color, markersize=4)
        line_artists.append((line, coords))
        point_artists.append((point, coords))

    if len(track_data) > 1:
        ax.legend(loc='upper left')

    total_frames = len(next(iter(track_data.values()))['x'])

    def update(frame_index):
        for line, coords in line_artists:
            line.set_data(coords['x'][:frame_index + 1], coords['y'][:frame_index + 1])
            line.set_3d_properties(coords['z'][:frame_index + 1])
        for point, coords in point_artists:
            point.set_data(np.asarray([coords['x'][frame_index]]), np.asarray([coords['y'][frame_index]]))
            point.set_3d_properties(np.asarray([coords['z'][frame_index]]))
        return [artist for artist, _ in line_artists] + [artist for artist, _ in point_artists]

    final_frame_index = max(total_frames - 1, 0)
    update(final_frame_index)
    if save_figure:
        fig.savefig(figure_output_path, bbox_inches='tight')

    if not export_movie:
        plt.close(fig)
        return None, str(figure_output_path) if save_figure else None

    animation = FuncAnimation(fig, update, frames=total_frames, interval=1000 / fps, blit=False)
    writer = FFMpegWriter(fps=fps, codec='libx264', extra_args=['-pix_fmt', 'yuv420p'])
    animation.save(output_path, writer=writer)
    plt.close(fig)
    display(Video(str(output_path), embed=True))
    return str(output_path), str(figure_output_path) if save_figure else None

In [ ]:
plot_fish_tracks(track_data, start_frame, end_frame, body_part=body_part)

In [ ]:
if GENERATE_MOVIE or SAVE_FIGURE:
    movie_path, figure_path = create_track_exports(
        track_data=track_data,
        source_file_path=file_path,
        start_frame=start_frame,
        end_frame=end_frame,
        body_part=body_part,
        fps=fps,
        save_figure=SAVE_FIGURE,
        export_movie=GENERATE_MOVIE,
    )
    if movie_path is not None:
        print(f'Saved movie to {movie_path}')
    if figure_path is not None:
        print(f'Saved figure snapshot to {figure_path}')
else:
    print('Skipping exports. Set SAVE_FIGURE = True in Cell 2 to save a PNG snapshot or GENERATE_MOVIE = True to render an MP4.')